# Tutorial 4.1: Thickness Optimization (Finite Difference)

In this tutorial, we will optimize the thickness of a 2D cooling panel using a naive Finite Difference approach to approximate the gradients that drive the Generalized Method of Moving Asymptotes (GCMMA) optimizer.

We will run the optimization in two distinct parts to build complexity:
1. **Uniform Thickness (1 Variable)**: We optimize a single global thickness variable to minimize the average temperature while strictly enforcing a maximum stress constraint.
2. **Multi-Variable Spatial Thickness (9 Variables)**: We optimize a 3x3 grid of control points defining a biquadratic thickness field to minimize average Von Mises stress, enforcing a volume fraction constraint.

## 1. Standard Imports and Setup

We begin by importing Firedrake and our optimization and visualization libraries. 

*Note: Ensure the `mma.py` file is present in your working directory, as Firedrake does not include MMA natively.*

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML, display
from tqdm.auto import tqdm

try:
    from mma import gcmmasub, asymp, concheck, raaupdate
except ImportError:
    print("Error: `mma.py` not found. Please upload it to your workspace.")

## 2. Mesh and Global Spaces

We set up our base quadrilateral mesh and define the common function spaces used throughout the notebook.

In [ ]:
# Mandatory Quadrilateral Elements
mesh = RectangleMesh(50, 50, 1.0, 1.0, quadrilateral=True)

# Function spaces for Temperature, Displacement, and Stress
V_T = FunctionSpace(mesh, "CG", 1)
V_u = VectorFunctionSpace(mesh, "CG", 1)
V_dg0 = FunctionSpace(mesh, "DG", 0)

## 3. Part 1: Uniform Thickness Optimization (1 Variable)

We define a single design variable representing the normalized global thickness of the panel. Our objective is to minimize average temperature subject to a stress penalty.

In [ ]:
# Design variable: Normalized Thickness [0, 1]
V_t = FunctionSpace(mesh, "Real", 0)
t_var = Function(V_t, name="Design_Variable").assign(0.5)

T = Function(V_T, name="Temperature")
u = Function(V_u, name="Displacement")
vm_stress = Function(V_dg0, name="VonMisesStress")

v_T, v_u = TestFunction(V_T), TestFunction(V_u)
T_trial, u_trial = TrialFunction(V_T), TrialFunction(V_u)

# Material Constants
k_cond = Constant(10.0)
T_ref = Constant(20.0)       
h_int = Constant(1000.0)       
T_source = Constant(1000.0)    
E = Constant(70e9)           
nu = Constant(0.3)           
alpha = Constant(1e-5)       
sigma_y = Constant(3.5e8)    
k_spring = Constant(1e9)       
t_min, t_max = Constant(0.001), Constant(1.0)          
c_pen = Constant(1e6) 
domain_area = assemble(Constant(1.0) * dx(mesh))

# Effective Material Properties (Scaled by physical thickness)
t_phys = t_min + t_var * (t_max - t_min)
k_eff = k_cond * t_phys
E_eff = E * t_phys
mu_eff = E_eff / (2 * (1 + nu))
lmbda_eff = (E_eff * nu) / ((1 + nu) * (1 - 2 * nu))

# Boundary Conditions and Heat Source
bc_u = []
bc_T = DirichletBC(V_T, T_ref, "on_boundary")
x, y = SpatialCoordinate(mesh)
h_local = h_int * exp(-15.0 * ((x - 0.5)**2 + (y - 0.5)**2))

def epsilon(vec):
    return sym(grad(vec))

def sigma(vec, temp):
    thermal_stress = (3 * lmbda_eff + 2 * mu_eff) * alpha * (temp - T_ref)
    return lmbda_eff * div(vec) * Identity(2) + 2 * mu_eff * epsilon(vec) - thermal_stress * Identity(2)

s_ufl = sigma(u, T)
vm_expr_ufl = sqrt(s_ufl[0,0]**2 + s_ufl[1,1]**2 - s_ufl[0,0]*s_ufl[1,1] + 3*s_ufl[0,1]**2 + 1e-8)
stress_violation = max_value(0.0, vm_stress / sigma_y - 1.0)
penalty_term = c_pen * stress_violation**2

# Solvers
thermal_form = (k_eff * inner(grad(T_trial), grad(v_T)) * dx + h_local * (T_trial - T_source) * v_T * dx)
prob_T = LinearVariationalProblem(lhs(thermal_form), rhs(thermal_form), T, bcs=bc_T)
solver_T = LinearVariationalSolver(prob_T)

mech_form = (inner(sigma(u_trial, T), epsilon(v_u)) * dx + k_spring * inner(u_trial, v_u) * ds)
prob_u = LinearVariationalProblem(lhs(mech_form), rhs(mech_form), u, bcs=bc_u)
solver_u = LinearVariationalSolver(prob_u)

def evaluate_objective_1():
    vm_stress.interpolate(vm_expr_ufl)
    temp_obj = assemble(T * dx) / domain_area
    pen_obj = assemble(penalty_term * dx)
    return temp_obj, pen_obj

solver_T.solve(); solver_u.solve()
J_temp_init, J_pen_init = evaluate_objective_1()
obj_scale_1 = 1.0 / float(J_temp_init)

history_temp, history_stress = [], []

## 4. The Naive GCMMA Loop (1 Variable)

We manually calculate the gradient for the single design variable by perturbing it with `h_fd` and re-solving the system.

In [ ]:
max_iter = 50
n, m = 1, 1 
xval = np.array([[0.5]])
xmin, xmax = np.array([[0.0]]), np.array([[1.0]])
h_fd = 1e-6 

xold1, xold2 = xval.copy(), xval.copy()
low, upp = xmin.copy(), xmax.copy()
a0, epsimin = 1.0, 1e-7
a, c, d = np.zeros((m, 1)), np.ones((m, 1)) * 5000.0, np.ones((m, 1))
raa0, raa0eps = 0.01, 1e-6
raa, raaeps = np.ones((m, 1)) * 0.01, np.ones((m, 1)) * 1e-6

pbar = tqdm(range(1, max_iter + 1), desc="1-Var FD Optimization")
for i in pbar:
    t_var.assign(float(xval[0, 0]))
    solver_T.solve()
    solver_u.solve()
    J_temp, J_pen = evaluate_objective_1()
    f0_base = (J_temp * obj_scale_1) + J_pen

    t_var.assign(float(xval[0, 0] + h_fd))
    solver_T.solve()
    solver_u.solve()
    J_temp_p, J_pen_p = evaluate_objective_1()
    f0_perturbed = (J_temp_p * obj_scale_1) + J_pen_p
    
    df0dx_val = (f0_perturbed - f0_base) / h_fd
    
    t_var.assign(float(xval[0, 0]))
    history_temp.append(float(J_temp))
    history_stress.append(float(vm_stress.dat.data_ro.max()))
    
    f0val, df0dx = np.array([[float(f0_base)]]), np.array([[float(df0dx_val)]])
    fval, dfdx = np.array([[0.0]]), np.array([[0.0]])
    
    low, upp, raa0, raa = asymp(i, n, xval, xold1, xold2, xmin, xmax, low, upp, raa0, raa, raa0eps, raaeps, df0dx, dfdx)
    xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx, fval, dfdx, a0, a, c, d)
    
    innerit = 0
    while innerit < 5:
        t_var.assign(float(xmma[0, 0]))
        solver_T.solve(); solver_u.solve()
        J_temp_new, J_pen_new = evaluate_objective_1()
        f0valnew, fvalnew = np.array([[float((J_temp_new * obj_scale_1) + J_pen_new)]]), np.array([[0.0]])
        
        if concheck(m, epsimin, f0app, f0valnew, fapp, fvalnew):
            break
        innerit += 1
        raa0, raa = raaupdate(xmma, xval, xmin, xmax, low, upp, f0valnew, fvalnew, f0app, fapp, raa0, raa, raa0eps, raaeps, epsimin)
        xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx, fval, dfdx, a0, a, c, d)
        
    xold2, xold1, xval = xold1.copy(), xval.copy(), xmma.copy()
    t_phys_curr = float(t_min.values()[0] + xval[0,0] * (t_max.values()[0] - t_min.values()[0]))
    
    pbar.set_postfix({'TempObj': f"{float(J_temp):.2e}", 't_opt': f"{t_phys_curr:.4f}"})

## 5. Results for Uniform Thickness

We visualize the optimization trajectory, plotting the temperature convergence and tracking the maximum stress to ensure it remains below the threshold.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history_temp, 'o-', color='tab:blue')
ax1.set_title("Avg Temp History (1-Var)")
ax1.set_xlabel("Iteration"); ax1.set_ylabel("Temperature [C]")
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(history_stress, 's-', color='tab:red')
ax2.axhline(y=float(sigma_y), color='black', linestyle='--')
ax2.set_title("Max Stress History (1-Var)")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Stress [Pa]")
ax2.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Part 2: Multi-Variable Thickness Optimization (9 Variables)

Next, we use a 3x3 grid of control points (9 variables) to define a biquadratic thickness field. We apply a naive Finite Difference approach to evaluate the gradients of the Von Mises stress objective and the volume fraction constraint. 

In [ ]:
# Re-initialize fields for Part 2
T_2 = Function(V_T, name="Temperature")
u_2 = Function(V_u, name="Displacement")
vm_stress_2 = Function(V_dg0, name="VonMisesStress")
phi_viz = Function(V_T, name="VolumeFraction")

# Define 9 control variables for a biquadratic fit
initial_phi = [0.8, 0.2, 0.8, 
               0.2, 0.8, 0.2, 
               0.8, 0.2, 0.8]
c_vars = [Constant(initial_phi[i]) for i in range(9)]
eps_phi = Constant(1e-4)

# Construct spatial volume fraction field
def N0(t): return 2 * (t - 0.5) * (t - 1.0)
def N1(t): return -4 * t * (t - 1.0)
def N2(t): return 2 * t * (t - 0.5)
basis = [N0, N1, N2]

phi = 0
for i in range(3):
    for j in range(3):
        phi += c_vars[i*3 + j] * basis[i](x) * basis[j](y)
phi_eff = eps_phi + (1.0 - eps_phi) * phi

# Material parameter scaling (Porous material penalization)
E_eff_2 = E * (phi_eff ** 1.0)
alpha_eff_2 = alpha * (phi_eff ** 1.0)
k_eff_2 = k_cond * phi_eff
mu_eff_2 = E_eff_2 / (2 * (1 + nu))
lmbda_eff_2 = (E_eff_2 * nu) / ((1 + nu) * (1 - 2 * nu))

# Set Temperature as a prescribed Gaussian (Uncoupled heat for this scenario)
x_c, y_c = Constant(0.5), Constant(0.5)
w_gauss = Constant(0.2)
T_2.interpolate(T_ref + (T_source - T_ref) * exp(-((x - x_c)**2 + (y - y_c)**2) / (2 * w_gauss**2)))

# Create nullspace for unconstrained boundaries
ns_basis = [
    Function(V_u).interpolate(Constant((1, 0))),
    Function(V_u).interpolate(Constant((0, 1))),
    Function(V_u).interpolate(as_vector([-y, x]))
]
nullspace_2 = VectorSpaceBasis(ns_basis)
nullspace_2.orthonormalize()

def sigma_2(vec, temp):
    thermal_stress = (3 * lmbda_eff_2 + 2 * mu_eff_2) * alpha_eff_2 * (temp - T_ref)
    return lmbda_eff_2 * div(vec) * Identity(2) + 2 * mu_eff_2 * epsilon(vec) - thermal_stress * Identity(2)

s_ufl_2 = sigma_2(u_2, T_2)
vm_expr_ufl_2 = sqrt(s_ufl_2[0,0]**2 + s_ufl_2[1,1]**2 - s_ufl_2[0,0]*s_ufl_2[1,1] + 3*s_ufl_2[0,1]**2 + 1e-8)

# Mechanical Solver Setup
mech_form_2 = (inner(sigma_2(u_trial, T_2), epsilon(v_u)) * dx)
prob_u_2 = LinearVariationalProblem(lhs(mech_form_2), rhs(mech_form_2), u_2, bcs=[])
solver_u_2 = LinearVariationalSolver(prob_u_2, nullspace=nullspace_2)

def evaluate_objective_2():
    return assemble(vm_expr_ufl_2 * dx) / domain_area

solver_u_2.solve()
J_init_2 = evaluate_objective_2()
obj_scale_2 = 1.0 / J_init_2

## 7. The Multi-Variable GCMMA Loop

We construct the GCMMA loop for the 9 variables. To calculate the gradients, we explicitly perturb each of the 9 variables sequentially with `h_fd` inside the loop.

In [ ]:
max_iter_2 = 25
n, m = 9, 1 
xval = np.array(initial_phi).reshape((n, 1))
xmin, xmax = np.zeros((n, 1)), np.ones((n, 1))
target_volume = 0.5

xold1, xold2 = xval.copy(), xval.copy()
low, upp = xmin.copy(), xmax.copy()
a0, epsimin = 1.0, 1e-6
a, c, d = np.ones((m, 1)), np.ones((m, 1)) * 1000.0, np.ones((m, 1))
raa0, raa0eps = 0.01, 1e-6
raa, raaeps = np.ones((m, 1)) * 0.01, np.ones((m, 1)) * 1e-6

history_obj_2, history_vol_2 = [], []
history_phi = []
history_vm = []
h_fd = 1e-6

pbar_2 = tqdm(range(1, max_iter_2 + 1), desc="9-Var FD Optimization")
for i in pbar_2:
    for j in range(n):
        c_vars[j].assign(float(xval[j, 0]))
        
    solver_u_2.solve()
    J_val = evaluate_objective_2() * obj_scale_2
    
    # Capture fields for later animation
    history_phi.append(phi_viz.interpolate(phi).dat.data.copy())
    vm_stress_2.interpolate(vm_expr_ufl_2)
    history_vm.append(vm_stress_2.dat.data.copy())

    history_obj_2.append(float(J_val))
    current_vol = assemble(phi * dx)
    history_vol_2.append(float(current_vol))
    fval = np.array([[1.0 - (current_vol / target_volume)]])
    
    # Compute Finite Difference Gradients
    df0dx_val = np.zeros((n, 1))
    dfdx = np.zeros((m, n))
    for j in range(n):
        c_vars[j].assign(float(xval[j, 0] + h_fd))
        solver_u_2.solve()
        
        J_p_scaled = evaluate_objective_2() * obj_scale_2
        df0dx_val[j, 0] = (J_p_scaled - J_val) / h_fd
        
        vol_p = assemble(phi * dx)
        dfdx[0, j] = -((vol_p - current_vol) / target_volume) / h_fd
        
        c_vars[j].assign(float(xval[j, 0]))
    
    f0val = np.array([[float(J_val)]])
    
    low, upp, raa0, raa = asymp(i, n, xval, xold1, xold2, xmin, xmax, low, upp, raa0, raa, raa0eps, raaeps, df0dx_val, dfdx)
    xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx_val, fval, dfdx, a0, a, c, d)

    innerit = 0
    while innerit < 5:
        for j in range(n):
            c_vars[j].assign(float(xmma[j, 0]))
            
        solver_u_2.solve()
        J_new_scaled = evaluate_objective_2() * obj_scale_2
        vol_new = assemble(phi * dx)
        f0valnew = np.array([[float(J_new_scaled)]])
        fvalnew = np.array([[1.0 - (vol_new / target_volume)]])
        
        if concheck(m, epsimin, f0app, f0valnew, fapp, fvalnew):
            break
            
        innerit += 1
        raa0, raa = raaupdate(xmma, xval, xmin, xmax, low, upp, f0valnew, fvalnew, f0app, fapp, raa0, raa, raa0eps, raaeps, epsimin)
        xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx_val, fval, dfdx, a0, a, c, d)
        
    xold2, xold1, xval = xold1.copy(), xval.copy(), xmma.copy()
    pbar_2.set_postfix({'StressObj': f"{float(J_val):.4f}", 'Volume': f"{float(current_vol):.4f}"})


## 8. Animation and Results for Multi-Variable Optimization

We dynamically reconstruct the progression of the material distribution and the resultant stress fields into an HTML animation to visualize the design evolution.

In [ ]:
# Convergence Plot for Multi-Variable Optimization (Part 2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Scaled Objective (Von Mises Stress)
ax1.plot(range(1, len(history_obj_2) + 1), history_obj_2, 'o-', color='darkblue', markersize=4)
ax1.set_title("Stress Objective Convergence")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Scaled Avg Von Mises Stress")
ax1.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Volume Fraction Constraint
ax2.plot(range(1, len(history_vol_2) + 1), history_vol_2, 's-', color='forestgreen', markersize=4)
ax2.axhline(y=target_volume, color='red', linestyle='--', label=f'Target Vol ({target_volume})')
ax2.set_title("Volume Fraction History")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Volume Fraction")
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

fig_anim, (ax_geom, ax_vm, ax_temp) = plt.subplots(1, 3, figsize=(18, 5))
ax_geom.set_title("Volume Fraction")
ax_vm.set_title("Von Mises Stress")
ax_temp.set_title("Temperature (Fixed)")

frames = []

for p_data, v_data in zip(history_phi, history_vm):
    phi_viz.dat.data[:] = p_data
    vm_stress_2.dat.data[:] = v_data
    
    c1 = tripcolor(phi_viz, axes=ax_geom, cmap='viridis')
    c2 = tripcolor(vm_stress_2, axes=ax_vm, cmap='jet')
    c3 = tripcolor(T_2, axes=ax_temp, cmap='inferno') # Temperature field
    
    frames.append([c1, c2, c3])

plt.close(fig_anim)
ani = ArtistAnimation(fig_anim, frames, interval=150, blit=True)
display(HTML(f'<div style=\"width:100%;\">{ani.to_jshtml()}</div>'))

## 9. Summary

In this tutorial, we structured two Finite Difference-driven topology optimizations:
* **Uniform Optimization**: Altering a single global thickness parameter verified the basic solver loop and constraint boundaries.
* **Multi-Variable Spatial Optimization**: Utilizing a 3x3 array of control variables mapping to a continuous biquadratic function space illustrated how geometry can evolve intelligently to lower stress limits.

## 10. Try It Yourself

**Alter the Target Mass Fraction**
Change the `target_volume` in the multi-variable loop from 0.5 to 0.3 or 0.7.
* *Prediction*: Tightening the volume constraint will force the algorithm to sacrifice structural stability, redistributing material more drastically to balance the severe thermal stresses against the constrained mass budget.

In [ ]:
# ==========================================
# Your Sandbox Workspace
# ==========================================

# Copy the relevant parts of the code from the sections above 
# and modify them here!

# Write your modified solver and plotting code below:
